# DATA209 — Advanced Exploratory Data Analysis
# Practical P29-30 · Full pipeline — industry-style notebook

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 15 · Module 4 · CO1-CO4

---

**Objective.** Assemble every step into one leak-free pipeline, validate it honestly, and produce the report structure expected in the final project.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.


### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

This session rebuilds everything from the raw file by design — a pipeline must be self-contained. Two summary values from earlier sessions are recomputed for the report.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

# --- from P15-16
df_clean = df.drop_duplicates().reset_index(drop=True)
for c in ["VisitorType", "Month"]:
    df_clean[c] = df_clean[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

model_cols = ["Administrative", "Administrative_Duration", "Informational",
              "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
              "BounceRates", "ExitRates", "PageValues"]

# --- from P23-24: which columns were skewed
skew_before = df_clean[model_cols].skew().sort_values(ascending=False)
needs = skew_before[skew_before.abs() > 1].index.tolist()
df_t = df_clean.copy()
for c in needs:
    df_t[c] = np.log1p(df_t[c].clip(lower=0))

# --- from P11-12: the silhouette verdict quoted in the final report
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
_Xs = StandardScaler().fit_transform(df_clean[model_cols])
_lab = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit_predict(_Xs)
overall = silhouette_score(_Xs, _lab, sample_size=4000, random_state=RANDOM_STATE)
print(f"silhouette at k=3: {overall:.3f}")

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P29-30 — Full pipeline — industry-style notebook

### Full pipeline implementation

Everything so far, assembled in the correct order. The single governing rule:

> **Split first. Fit every transformer on the training data only.**

A `Pipeline` enforces this automatically — which is why it is the required structure for the
final project.

In [ ]:
# ---- Start again from the raw file, so the pipeline is self-contained ---
raw = pd.read_csv(find("online_shoppers_intention.csv")).drop_duplicates().reset_index(drop=True)
print("Raw, deduplicated:", raw.shape)

TARGET = "Revenue"
y_all = raw[TARGET].astype(int)
X_all = raw.drop(columns=[TARGET])

# --- feature construction expressed as a reusable function -----------------
def engineer(frame):
    f = frame.copy()
    f["total_pages"]    = f["Administrative"] + f["Informational"] + f["ProductRelated"]
    f["total_duration"] = (f["Administrative_Duration"] + f["Informational_Duration"]
                           + f["ProductRelated_Duration"])
    f["avg_time_per_page"] = (f["total_duration"] / f["total_pages"].replace(0, np.nan)).fillna(0)
    f["product_focus"]     = (f["ProductRelated"] / f["total_pages"].replace(0, np.nan)).fillna(0)
    f["exit_bounce_gap"]   = f["ExitRates"] - f["BounceRates"]
    f["has_page_value"]    = (f["PageValues"] > 0).astype(int)
    f["is_holiday_month"]  = f["Month"].isin(["Nov", "Dec"]).astype(int)
    return f

X_all = engineer(X_all)
print("After feature construction:", X_all.shape)

In [ ]:
# ---- Split BEFORE anything is fitted ------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE)

print(f"Train {X_train.shape}   Test {X_test.shape}")
print(f"Positive rate — train {y_train.mean()*100:.2f}%  test {y_test.mean()*100:.2f}%")
print("Stratified, so both sides carry the same class balance.")

In [ ]:
# ---- Build the preprocessing + model pipeline ---------------------------
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = ["Month", "VisitorType", "OperatingSystems",
                        "Browser", "Region", "TrafficType", "Weekend"]
categorical_features = [c for c in categorical_features if c in X_train.columns]

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("log",    FunctionTransformer(lambda a: np.log1p(np.clip(a, 0, None)),
                                   feature_names_out="one-to-one")),
    ("scale",  StandardScaler()),
])

categorical_pipe = Pipeline([
    # Cast to plain strings first: these columns mix integer codes with text labels,
    # and an imputer will otherwise try to read 'Dec' as a number.
    ("as_text", FunctionTransformer(lambda d: pd.DataFrame(d).astype(str),
                                    feature_names_out="one-to-one")),
    ("impute",  SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", drop="first",
                              min_frequency=0.01, sparse_output=False)),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
], remainder="drop")

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=3000, class_weight="balanced",
                                 random_state=RANDOM_STATE)),
])

print(f"Numeric features    : {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print("\nEvery fitted step lives inside the pipeline. Nothing is fitted before the split.")

In [ ]:
# ---- Cross-validate on the training data only ---------------------------
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             RocCurveDisplay, PrecisionRecallDisplay)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for metric in ["roc_auc", "average_precision", "f1"]:
    s = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring=metric, n_jobs=-1)
    print(f"{metric:>18}: {s.mean():.4f} +/- {s.std():.4f}")

print("\nAccuracy is deliberately absent: at 15% positives it rewards predicting 'never'.")
print("ROC-AUC and average precision (PR-AUC) are the honest choices here.")

In [ ]:
# ---- Compare against a second model, then evaluate ONCE on the test set -
rf_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                     random_state=RANDOM_STATE, n_jobs=-1)),
])

for name, p in [("logistic regression", pipeline), ("random forest", rf_pipeline)]:
    s = cross_val_score(p, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"{name:>22}: CV ROC-AUC {s.mean():.4f} +/- {s.std():.4f}")

# the test set is touched exactly once, at the very end
final = rf_pipeline.fit(X_train, y_train)
proba = final.predict_proba(X_test)[:, 1]
pred  = final.predict(X_test)

print(f"\nHELD-OUT TEST SET (used once)")
print(f"  ROC-AUC          : {roc_auc_score(y_test, proba):.4f}")
print(f"  Average precision: {average_precision_score(y_test, proba):.4f}")
print("\n", classification_report(y_test, pred, target_names=["no purchase", "purchase"]))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0], color="#6B4C7A")
axes[0].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1], color="#6B4C7A")
axes[1].set_title("Precision-recall curve")
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt=",d", cmap="Purples",
            ax=axes[2], cbar=False,
            xticklabels=["pred no", "pred yes"], yticklabels=["true no", "true yes"])
axes[2].set_title("Confusion matrix")
plt.tight_layout(); plt.show()

### Leakage audit

Four paths, each checked explicitly. **Document this check in your final project.**

In [ ]:
# ---- Leakage audit ------------------------------------------------------
audit_rows = [
    dict(path="1. Preprocessing fitted on all data",
         risk="Scaler/imputer sees test statistics",
         status="CLOSED",
         evidence="All transformers live inside the Pipeline; fitted per fold on train only"),
    dict(path="2. Target leakage",
         risk="A feature encodes the answer",
         status="CHECKED",
         evidence="All features derive from session behaviour observable before checkout"),
    dict(path="3. Temporal leakage",
         risk="Training on rows recorded after test rows",
         status="NOTED",
         evidence="Month is available but no row-level timestamp; a random split is used. "
                  "With timestamps, a time-based split would be required"),
    dict(path="4. Duplicate leakage",
         risk="Same entity in train and test",
         status="CLOSED",
         evidence="drop_duplicates() applied before the split"),
]
print(pd.DataFrame(audit_rows).to_string(index=False))

# a blunt but effective check: is any single feature implausibly predictive?
single = {}
for c in numeric_features:
    try:
        single[c] = roc_auc_score(y_train, X_train[c])
    except Exception:
        pass
suspect = pd.Series(single).sort_values(ascending=False).head(5)
print("\nSingle-feature ROC-AUC (top 5) — anything above ~0.95 suggests leakage:")
print(suspect.round(4).to_string())
print("Highest is well below that threshold, so no feature is standing in for the target.")

### Industry-style notebook submission

The structure below is the required shape of the final project report. Each section answers one
question a reader will actually ask.

In [ ]:
# ---- Final report skeleton ----------------------------------------------
report = f"""
================================================================================
DATA209 FINAL PROJECT — EXPLORATORY DATA ANALYSIS REPORT
Dataset: Online Shoppers Purchasing Intention (UCI) | {raw.shape[0]:,} sessions x {raw.shape[1]} columns
================================================================================

1. WHAT THE DATA IS
   Source          : UCI Machine Learning Repository
   Unit of analysis: one browsing session
   Sampling frame  : sessions on a single retailer over a 12-month period
   Period          : {sorted(raw['Month'].unique())}
   Target          : Revenue — {y_all.mean()*100:.1f}% positive
   Cannot speak to : other retailers, other periods, or offline behaviour

2. WHAT WAS WRONG WITH IT
   Duplicates removed  : {12330 - len(raw):,} exact duplicate rows
   Explicit nulls      : {raw.isna().sum().sum()}
   Disguised missing   : none found in this dataset (contrast with pima.csv)
   Strongly skewed     : {len(needs)} of {len(model_cols)} numeric columns, log1p applied
   Rare categories     : levels below 1% grouped into 'Other' at encoding
   Validity violations : none against the domain rules in P15-16

3. WHAT IT SHOWS
   - PageValues is the strongest single separator of converting sessions
   - Returning visitors convert at a different rate from new visitors
   - Conversion is seasonal, rising into the November-December period
   - BounceRates and ExitRates are near-duplicates (multicollinear); keep one
   - K-means finds no strong natural segmentation (silhouette {overall:.2f})

4. WHAT COMES NEXT
   H1  Sessions reaching a positive-PageValue page convert far more often
       falsified by: no difference once session depth is controlled for
   H2  Returning visitors convert more because of prior evaluation
       falsified by: difference vanishes after conditioning on month and traffic type
   H3  Conversion peaks in the holiday period
       falsified by: monthly rates within sampling variation of the overall rate

   Modelling-ready dataset: {len(numeric_features)} numeric + {len(categorical_features)} categorical
   Recommended next step  : threshold-tuned classifier optimised for precision at
                            fixed recall, since the cost of a wasted contact is low
                            and the cost of a missed purchase is high

5. LIMITATIONS
   - No user identifier, so repeat visitors cannot be linked across sessions
   - No row-level timestamp, so a strict temporal split is impossible
   - Promotional calendar is unobserved and confounds the seasonal finding
================================================================================
"""
print(report)

### Deliverable — P29-30 and the final project

An industry-style notebook plus a 12-minute presentation, worth 30% of the course.

**Submission checklist**

- [ ] Kernel restarted and *Run All* completes without error
- [ ] Relative paths only — no `C:\Users\...`
- [ ] `random_state` set everywhere it exists
- [ ] Split occurs before any transformer is fitted
- [ ] All preprocessing inside a `Pipeline`
- [ ] Test set evaluated exactly once
- [ ] Leakage audit table included
- [ ] Metric appropriate to the class imbalance — not accuracy
- [ ] Every figure carries a sentence saying what it shows
- [ ] Limitations section states what the data cannot support